# ADY201m — Analysis with Python: RQ1, RQ2, RQ3

**Đáp ứng rubric mục 5:** Notebook có thể chạy lại trên **18 CSV đã làm sạch**; kiểm tra dữ liệu, phân tích bằng Pandas, đối chiếu kết quả pipeline FULL, biểu đồ và kết luận kèm giới hạn.

- **RQ1:** Phân tích Pearson trực tiếp từ các cột điểm trước FINAL EXAM theo từng nguồn.
- **RQ2:** Đọc `KET_QUA_CHAY_V1_1/RQ2_model_comparison.csv` sinh từ **lần train đầy đủ**, tính kiểm tra F2 và xếp hạng **5 mô hình**.
- **RQ3:** Đọc kết quả FULL `RQ3_four_scenarios.csv`, so sánh baseline/SMOTE có và không tuning (tránh đánh đồng tác dụng của SMOTE với tuning).

**Quan trọng:** Notebook này dùng Pandas để tính RQ1 và **kiểm chứng/diễn giải** RQ2, RQ3 từ outputs thật. Để huấn luyện 5 thuật toán và áp dụng SMOTE thực tế, chạy file `ADY201m_RQ1_RQ2_RQ3_PIPELINE.py`. SQL truy vấn các bảng dữ liệu và metrics tương ứng, không chạy ML.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chạy notebook từ thư mục gốc của ZIP sau giải nén.
ROOT = Path.cwd()
if not (ROOT/'DU_LIEU'/'18_CSV_GOC').exists():
    possible = list(ROOT.glob('*/DU_LIEU/18_CSV_GOC'))
    if len(possible) == 1:
        ROOT = possible[0].parent.parent
assert (ROOT/'DU_LIEU'/'18_CSV_GOC').exists(), 'Hãy mở notebook trong thư mục gốc chứa DU_LIEU.'
DATA = ROOT/'DU_LIEU'/'18_CSV_GOC'
RESULTS = ROOT/'KET_QUA_CHAY_V1_1'
OUT = ROOT/'KET_QUA_PANDAS_DEMO'
OUT.mkdir(exist_ok=True)
print('ROOT:',ROOT)

ROOT: /tmp/claude-0/-home-claude/86b98766-2cfc-5df3-accc-f3ea7128e976/scratchpad/pkg/ADY201m_DEMO_SQL_PANDAS_RQ123_V1_1


## 0. Data understanding và kiểm tra dữ liệu

**Nguồn:** 18 CSV cleaned từ pipeline, **không lấy số liệu RQ1 từ file combined**. Ghép bảng theo tên cột với `concat`, vì mỗi nguồn có các nhóm điểm khác nhau. `TARGET=1` là kết quả không đạt; lọc `HAS_PREFINAL=True` giống SQL `v_ml_prefinal_dataset`.

Các cột `FINAL EXAM`, `TOTAL`, `STATUS`, `STATUS_CLEAN` hoặc `TARGET` **không được dùng làm predictor** (rò rỉ thông tin). `TARGET` chỉ làm nhãn so sánh.

In [2]:
files = sorted(DATA.glob('*_cleaned.csv'))
assert len(files) == 18, f'Phải có đúng 18 CSV, hiện có {len(files)}'
parts = []
for file in files:
    part = pd.read_csv(file, low_memory=False)
    required = {'SOURCE_FILE', 'ROW_IDX_ORIG', 'CODE', 'TARGET', 'HAS_PREFINAL'}
    assert required.issubset(part.columns), f'{file.name} thiếu cột {required-set(part.columns)}'
    parts.append(part)
data = pd.concat(parts, ignore_index=True, sort=False)
assert len(data) == 124791
assert data['SOURCE_FILE'].nunique() == 18
assert not data.duplicated(['SOURCE_FILE','ROW_IDX_ORIG']).any()
assert data['TARGET'].isin([0,1]).all()
eligible = data.loc[data['HAS_PREFINAL'].astype(bool)].copy()
assert len(eligible) == 110704
print('Toàn bộ:',len(data), 'dòng | 18 nguồn | đủ điều kiện:',len(eligible))
print('Phân bổ nhãn trước SMOTE:')
print(eligible['TARGET'].value_counts().sort_index().rename('n').to_frame().assign(pct=lambda x:100*x.n/len(eligible)).round(2))

Toàn bộ: 124791 dòng | 18 nguồn | đủ điều kiện: 110704
Phân bổ nhãn trước SMOTE:
            n   pct
TARGET             
0       81805  73.9
1       28899  26.1


## RQ1 — Thành phần trước final exam liên hệ mạnh nhất với không đạt?

**Phương pháp:** Với từng cặp `(SOURCE_FILE, component)`, bỏ điểm thiếu (pairwise complete), yêu cầu tối thiểu 30 cặp, điểm và TARGET đều có biến thiên; tính Pearson giữa điểm liên tục và TARGET nhị phân (point-biserial). Sau đó tính **trung vị |r| theo nhóm điểm** để tránh gộp sai các cấu trúc đánh giá khác nhau. Trình bày riêng độ phủ; một nhóm chỉ xuất hiện 2 file chưa đại diện tốt.

In [3]:
# Chỉ 21 biến điểm trước kỳ thi cuối kỳ dùng trong pipeline.
FEATURES = [
 'LABS-10','ASSIGNMENTS-10','PROGRESS TESTS-30','ASSIGNMENTS-30',
 'EXERCISES-30','QUIZ-20','ASSIGNMENTS-20','PRACTICAL EXAM-30',
 'PROGRESS TESTS-20','PROGRESS TESTS-10','QUIZ-10','LABS-15',
 'PRACTICAL EXAM-25','PRESENTATION-20','ACTIVE LEARNING-10','PROJECT-30',
 'EXERCISES-10','LABS-20','PRACTICAL EXAM-40','WORKSHOP-10','PRACTICAL EXAM-20'
]
assert len(FEATURES)==21 and set(FEATURES).issubset(eligible.columns)
per_file_rows=[]
for source, group in eligible.groupby('SOURCE_FILE'):
    for component in FEATURES:
        x = pd.to_numeric(group[component], errors='coerce')
        pair = pd.DataFrame({'score':x,'target':group['TARGET']}).dropna()
        if len(pair)<30 or pair.score.nunique()<2 or pair.target.nunique()<2:continue
        r = pair.score.corr(pair.target)
        if pd.notna(r):
            per_file_rows.append({'source_file':source, 'assessment_component':component,
                 'assessment_family':component.rsplit('-',1)[0], 'n':len(pair),
                 'pearson_r':r,'abs_r':abs(r)})
per_file = pd.DataFrame(per_file_rows)
rq1 = (per_file.groupby('assessment_family',as_index=False)
       .agg(distinct_files=('source_file','nunique'),component_file_pairs=('pearson_r','size'),
            median_abs_correlation=('abs_r','median'))
       .sort_values('median_abs_correlation',ascending=False))
known = pd.read_csv(RESULTS/'RQ1_cross_structure_consistency.csv')
check = rq1.merge(known[['assessment_family','median_abs_correlation']],on='assessment_family',suffixes=('_pandas','_pipeline'))
assert len(check)==len(rq1)==len(known)
assert np.allclose(check.median_abs_correlation_pandas,check.median_abs_correlation_pipeline,atol=1e-8)
print(rq1.round(6).to_string(index=False))
per_file.to_csv(OUT/'RQ1_pearson_per_source.csv',index=False)
rq1.to_csv(OUT/'RQ1_ranked_families.csv',index=False)
strong_broad=rq1.query('distinct_files >= 5').iloc[0]
print(f"RQ1 (>=5 files): {strong_broad.assessment_family}; median |r|={strong_broad.median_abs_correlation:.6f}; {int(strong_broad.distinct_files)} sources")
print('Lưu ý: PROJECT mạnh hơn về số tuyệt đối, nhưng chỉ có 2 file dữ liệu.')

assessment_family  distinct_files  component_file_pairs  median_abs_correlation
          PROJECT               2                     2                0.766637
   PRACTICAL EXAM              10                    13                0.719285
     PRESENTATION               3                     3                0.689721
  ACTIVE LEARNING               2                     2                0.676714
             LABS               6                     7                0.651801
   PROGRESS TESTS              12                    12                0.641883
      ASSIGNMENTS              12                    15                0.639703
        EXERCISES               2                     2                0.631092
             QUIZ               6                     6                0.617440
         WORKSHOP               1                     1                0.532896
RQ1 (>=5 files): PRACTICAL EXAM; median |r|=0.719285; 10 sources
Lưu ý: PROJECT mạnh hơn về số tuyệt đối, nhưng chỉ có 2

In [4]:
chart=rq1.query('distinct_files >= 5').sort_values('median_abs_correlation')
ax=chart.plot.barh(x='assessment_family',y='median_abs_correlation',figsize=(9,4),legend=False)
ax.set(title='RQ1 | Median absolute Pearson r by assessment family (>=5 sources)',xlabel='Median |r|',ylabel='')
plt.tight_layout();plt.savefig(OUT/'RQ1_correlation.png',dpi=140);plt.show()

## RQ2 — So sánh 5 mô hình

**Phương pháp:** Dùng kết quả **validation** để lựa chọn thuật toán, tránh chọn theo test gây thiên lệch. Chỉ số chính **F2-score** vì ưu tiên recall trong early-warning; cung cấp precision, recall và PR-AUC để bổ sung. Tính lại `F2 = 5 TP / (5 TP + 4 FN + FP)` bằng Pandas từ confusion matrix đã lưu. **Notebook không train lại mô hình**, phần train có trong pipeline Python đính kèm.

In [5]:
model = pd.read_csv(RESULTS/'RQ2_model_comparison.csv')
expected = {'Decision Tree','Random Forest','XGBoost','LightGBM','CatBoost'}
assert len(model)==5 and set(model.model)==expected
model['f2_recomputed'] = 5*model.val_tp / (5*model.val_tp+4*model.val_fn+model.val_fp)
assert np.allclose(model.val_f2, model.f2_recomputed,atol=1e-10)
model = model.sort_values('f2_recomputed',ascending=False).reset_index(drop=True)
print(model[['model','val_precision','val_recall','val_f2','val_pr_auc','test_f2']].round(6).to_string(index=False,na_rep='N/A'))
model.to_csv(OUT/'RQ2_5_models_checked.csv',index=False)
print(f'RQ2: {model.iloc[0].model}; validation F2 = {model.iloc[0].val_f2:.6f}')
print(f'     chênh với hạng 2 ({model.iloc[1].model}) = {model.iloc[0].val_f2-model.iloc[1].val_f2:.4f} (1 lần chia, 1 random_state; chỉ là bằng chứng mô tả)')

        model  val_precision  val_recall   val_f2  val_pr_auc  test_f2
Random Forest       0.848914    0.858603 0.856648    0.930897 0.840868
     LightGBM       0.944867    0.826235 0.847517    0.946399      N/A
      XGBoost       0.950256    0.823339 0.845936    0.946420      N/A
     CatBoost       0.952381    0.814310 0.838626    0.944406      N/A
Decision Tree       0.812900    0.843782 0.837419    0.783516      N/A
RQ2: Random Forest; validation F2 = 0.856648
     chênh với hạng 2 (LightGBM) = 0.0091 (1 lần chia, 1 random_state; chỉ là bằng chứng mô tả)


In [6]:
ax=model.sort_values('f2_recomputed').plot.barh(x='model',y='f2_recomputed',figsize=(9,4),legend=False)
ax.set(xlabel='Validation F2',ylabel='',title='RQ2 | Validation F2 across five algorithms',xlim=(0,1))
plt.tight_layout();plt.savefig(OUT/'RQ2_models.png',dpi=140);plt.show()

## RQ3 — SMOTE cùng F2-based threshold tuning cải thiện được không?

Bộ thử nghiệm lưu 4 phương án (baseline/SMOTE × ngưỡng cố định 0.50/tuning theo validation). Phải so sánh **SMOTE + tuning** với **Baseline + tuning** để phân lập tác động bổ sung của SMOTE. Chênh lệch F2 giữa hai nhánh trên **test** là bằng chứng mô tả, không phải kiểm định ý nghĩa thống kê. SMOTE được pipeline áp dụng **chỉ trên train**, không phải trước khi chia dữ liệu.

In [7]:
scenarios = pd.read_csv(RESULTS/'RQ3_four_scenarios.csv')
assert len(scenarios)==4
scenarios['precision_checked']=scenarios.tp/(scenarios.tp+scenarios.fp)
scenarios['recall_checked']=scenarios.tp/(scenarios.tp+scenarios.fn)
scenarios['f2_checked']=5*scenarios.tp/(5*scenarios.tp+4*scenarios.fn+scenarios.fp)
for metric in ('precision','recall','f2'):
    assert np.allclose(scenarios[metric],scenarios[metric+'_checked'],atol=1e-10)
print(scenarios[['scenario','threshold','precision_checked','recall_checked','f2_checked']].round(6).to_string(index=False))
scenarios.to_csv(OUT/'RQ3_4_scenarios_checked.csv',index=False)
s=scenarios.set_index('scenario')
b=s.loc['Baseline + F2 tuning']
m=s.loc['SMOTE + F2 tuning']
print(f"RQ3: delta F2 (SMOTE+tuning - Baseline+tuning) = {m.f2_checked-b.f2_checked:+.6f}")
print(f"     delta Recall = {m.recall_checked-b.recall_checked:+.6f}")
d_f2=m.f2_checked-b.f2_checked; d_rc=m.recall_checked-b.recall_checked
print(f"Kết luận (mô tả, 1 lần chia, chưa kiểm định thống kê): SMOTE+tuning {'cao hơn' if d_f2>0 else 'thấp hơn'} {abs(d_f2):.4f} F2 và {'cao hơn' if d_rc>0 else 'thấp hơn'} {abs(d_rc):.4f} recall so với baseline+tuning.")
if abs(d_f2)<0.005: print('Chênh lệch F2 rất nhỏ: không đủ cơ sở để kết luận SMOTE cải thiện (hay làm giảm) F2.')
u=s.loc['Baseline @ 0.50']
print(f"Chỉ tuning ngưỡng (baseline): F2 {b.f2_checked-u.f2_checked:+.6f}, recall {b.recall_checked-u.recall_checked:+.6f}, precision {b.precision_checked-u.precision_checked:+.6f}")

            scenario  threshold  precision_checked  recall_checked  f2_checked
     Baseline @ 0.50       0.50           0.844672        0.839923    0.840868
Baseline + F2 tuning       0.39           0.772146        0.859251    0.840293
        SMOTE @ 0.50       0.50           0.873862        0.826568    0.835613
   SMOTE + F2 tuning       0.34           0.794943        0.856264    0.843255
RQ3: delta F2 (SMOTE+tuning - Baseline+tuning) = +0.002962
     delta Recall = -0.002987
Kết luận (mô tả, 1 lần chia, chưa kiểm định thống kê): SMOTE+tuning cao hơn 0.0030 F2 và thấp hơn 0.0030 recall so với baseline+tuning.
Chênh lệch F2 rất nhỏ: không đủ cơ sở để kết luận SMOTE cải thiện (hay làm giảm) F2.
Chỉ tuning ngưỡng (baseline): F2 -0.000575, recall +0.019329, precision -0.072526


In [8]:
ax=scenarios.set_index('scenario')[['precision_checked','recall_checked','f2_checked']].plot.bar(
    figsize=(10,4.4),rot=15)
ax.set(ylim=(0,1),ylabel='Test metric',title='RQ3 | Four controlled scenarios (test set)')
plt.tight_layout();plt.savefig(OUT/'RQ3_scenarios.png',dpi=140);plt.show()

## Kết luận, kiểm chứng và giới hạn

- **RQ1:** PRACTICAL EXAM mạnh nhất **trong nhóm có độ phủ ít nhất 5/18 nguồn**; Pearson âm có nghĩa điểm cao gắn với xác suất không đạt thấp, nhưng không thể suy luận nhân quả. PROJECT có độ lớn |r| lớn hơn nhưng chỉ xuất hiện ở hai nguồn. Phân tích độ nhạy (`RQ1_SENSITIVITY.py`: trung vị trong từng file×nhóm rồi trung vị giữa các file) không đổi kết luận.
- **RQ2:** Trong lần chạy v1.1 (không imputation, `2026-11-04T07:40:03`), **Random Forest** có validation F2 cao nhất (0.8566), hơn LightGBM (0.8475). Đây là kết quả của một lần chia và một random_state; không khẳng định thuật toán nào vượt trội tổng quát. Kết quả các lần chạy cũ (v1.0, có imputation) không được trộn vào.
- **RQ3:** Threshold tuning làm recall tăng (baseline 0.8399 → 0.8593; SMOTE 0.8266 → 0.8563) và precision giảm; F2 của baseline gần như không đổi (0.8409 → 0.8403), còn nhánh SMOTE tăng nhẹ (0.8356 → 0.8433). SMOTE + tuning chênh baseline + tuning chỉ +0.0030 F2 và -0.0030 recall: **không có bằng chứng SMOTE cải thiện F2**. Dấu của chênh lệch này đổi chiều giữa các biến thể phân tích độ nhạy, nên không nên diễn giải theo chiều nào.

**Giới hạn:** 18 CSV này đã được clean (không thay thế pipeline làm sạch dữ liệu thô). RQ2 và RQ3 được đọc từ kết quả huấn luyện thật của pipeline v1.1; muốn tái lập cần chạy lại script và kiểm tra phiên bản thư viện (kết quả phụ thuộc phiên bản). Khoảng 86% ô predictor bị thiếu vì mỗi môn chỉ có một số thành phần điểm; pipeline không imputation (riêng nhánh SMOTE dùng median imputer vì SMOTE không xử lý NaN). `CODE` được hiểu là mã sinh viên **trong từng file môn học**; không giả định cùng `CODE` ở file khác là cùng sinh viên. 4.812 tổ hợp `(SOURCE_FILE, CODE, Mã học kỳ)` lặp lại đã có sẵn trong file Excel gốc (không do bước clean tạo ra), nhãn xung đột ở 3.317 tổ hợp; pipeline giữ tất cả các dòng và chia nhóm theo `(SOURCE_FILE, CODE)`. Các file CSV đã xuất ở `KET_QUA_PANDAS_DEMO/` hỗ trợ đối chiếu SQL.
